# Lab 5: Music Generation with LoRA Fine-tuning (ACE-Step 1.5)

Ukrainian folk music

Environment: vast.ai (RTX 3090 (24 GB VRAM) + Gradio UI)

## 1. GPU Check and installation

In [1]:
# Verify GPU
!nvidia-smi
import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()} — {torch.cuda.get_device_name(0)}')
free, total = torch.cuda.mem_get_info()
print(f'VRAM: {free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total')

Mon May  4 18:41:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.195.03             Driver Version: 570.195.03     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        On  |   00000000:84:00.0 Off |                  N/A |
| 44%   61C    P8             23W /  350W |       1MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Clone ACE-Step 1.5
from pathlib import Path

ACESTEP_DIR = Path("ACE-Step-1.5")
if not ACESTEP_DIR.exists():
    !git clone https://github.com/ace-step/ACE-Step-1.5.git
    print("Cloned.")
else:
    print("Already exists.")

Cloning into 'ACE-Step-1.5'...
remote: Enumerating objects: 12516, done.
remote: Counting objects: 100% (556/556), done.
remote: Compressing objects: 100% (157/157), done.
remote: Total 12516 (delta 472), reused 399 (delta 399), pack-reused 11960 (from 3)
Receiving objects: 100% (12516/12516), 13.20 MiB | 25.55 MiB/s, done.
Resolving deltas: 100% (8023/8023), done.
Cloned.


In [3]:
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()} — {torch.cuda.get_device_name(0)}')

!git clone --depth 1 https://github.com/ace-step/ACE-Step-1.5.git

!cd ACE-Step-1.5 && pip install ./acestep/third_parts/nano-vllm --no-deps -q
!cd ACE-Step-1.5 && pip install -e . --no-deps -q
!pip install torchaudio -q
!pip install "transformers>=4.51.0,<4.58.0" "diffusers>=0.37.0" -q
!pip install gradio==6.2.0 -q
!pip install loguru einops peft accelerate -q
!pip install soundfile scipy matplotlib toml modelscope -q
!pip install lightning lycoris-lora tensorboard -q
!pip install "vector-quantize-pytorch>=1.27.15" pytorch-wavelets pywavelets -q
!pip install "numba>=0.60.0" -q
!pip install fastapi "uvicorn[standard]" diskcache typer-slim -q

!pip cache purge

!python3 -c "import torchaudio; print('torchaudio OK')"
!cd ACE-Step-1.5 && python3 -c "from acestep.handler import AceStepHandler; print('ACE-Step OK')"

Mon May  4 18:41:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.195.03             Driver Version: 570.195.03     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        On  |   00000000:84:00.0 Off |                  N/A |
| 57%   66C    P0            124W /  350W |     262MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!pip install torch==2.6.0 torchaudio==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124 -q
!pip cache purge

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nano-vllm 0.2.0 requires xxhash, which is not installed.
ace-step 1.5.0 requires torchao<0.17.0,>=0.16.0; platform_machine != "aarch64", which is not installed.
ace-step 1.5.0 requires torchcodec>=0.9.1; platform_machine != "aarch64", which is not installed.
ace-step 1.5.0 requires torch==2.10.0+cu128; sys_platform == "linux" and platform_machine == "x86_64", but you have torch 2.6.0+cu124 which is incompatible.
ace-step 1.5.0 requires torchaudio==2.10.0+cu128; sys_platform == "linux" and platform_machine == "x86_64", but you have torchaudio 2.6.0+cu124 which is incompatible.
ace-step 1.5.0 requires torchvision==0.25.0+cu128; sys_platform == "linux" and platform_machine == "x86_64", but you have torchvision 0.21.0+cu124 which is incompatible.
Files removed: 48


In [5]:
import torch
print(torch.__version__)
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

2.11.0+cu128
True NVIDIA GeForce RTX 3090


## 2. Dataset Upload and validation

Uploaded dataset folder to the vast.ai instance.

In [7]:
import shutil

DATASET_SRC = Path("dataset")
ACESTEP_DATASET = ACESTEP_DIR / "dataset"

if DATASET_SRC.exists() and DATASET_SRC != ACESTEP_DATASET:
    if ACESTEP_DATASET.exists():
        shutil.rmtree(ACESTEP_DATASET)
    shutil.copytree(DATASET_SRC, ACESTEP_DATASET)
    print(f"Copied dataset to {ACESTEP_DATASET}")
elif ACESTEP_DATASET.exists():
    print(f"Dataset already at {ACESTEP_DATASET}")
else:
    print("No dataset found! Upload your dataset folder first.")

if ACESTEP_DATASET.exists():
    print(f"Files: {len(list(ACESTEP_DATASET.iterdir()))}")

Copied dataset to ACE-Step-1.5/dataset
Files: 34


In [8]:
import json

AUDIO_EXTS = {'.mp3', '.wav', '.flac', '.ogg', '.opus', '.m4a'}
audio_names = set()
for ext in AUDIO_EXTS:
    for f in ACESTEP_DATASET.glob(f'*{ext}'):
        audio_names.add(f.stem)


complete = 0
for name in sorted(audio_names):
    has_lyrics = (ACESTEP_DATASET / f'{name}.lyrics.txt').exists()
    has_json = (ACESTEP_DATASET / f'{name}.json').exists()
    ok = has_lyrics and has_json
    if ok: complete += 1

print(f"\n{complete}/{len(audio_names)} songs complete.")


11/11 songs complete.


In [9]:
for name in sorted(audio_names):
    json_path = ACESTEP_DATASET / f"{name}.json"
    if json_path.exists():
        with open(json_path) as f:
            ann = json.load(f)
        print(f"{name}")
        print(f"  BPM: {ann.get('bpm')} | Key: {ann.get('keyscale')} | Lang: {ann.get('language')}")
        print(f"  Caption: {ann.get('caption', '')[:80]}...")
        print()

I_am_walking
  BPM: 120 | Key: D major | Lang: uk
  Caption: A playful Ukrainian folk song with a humorous narrative about a lazy but attract...

I_have_dark_hair
  BPM: 118 | Key: E minor | Lang: uk
  Caption: A cheeky Ukrainian folk song about a dark-haired young woman and a forester, wit...

godfather_was_courting_godmother
  BPM: 108 | Key: G major | Lang: uk
  Caption: A traditional Ukrainian folk song about a godfather courting a godmother through...

here_godmother
  BPM: 142 | Key: A minor | Lang: uk
  Caption: A raucous Hutsul folk song with bawdy humor and a recurring demand-style chorus,...

mother_in_law_boots
  BPM: 128 | Key: D major | Lang: uk
  Caption: A humorous Ukrainian wedding folk song about a son-in-law washing his mother-in-...

mum_in_law_give_me_car
  BPM: 140 | Key: A major | Lang: uk
  Caption: An energetic and humorous Ukrainian folk-pop song about a son-in-law asking his ...

my_godmother_and_I_love_dancing
  BPM: 138 | Key: C major | Lang: uk
  Caption: A

## 3. Launch Gradio UI (Preprocess + Train + Generate)

With 24 GB VRAM on RTX 3090.

**Step-by-step in the Gradio UI:**

### Preprocessing
1. Click **"Initialize Service"** in Settings (acestep-v15-turbo, device: auto)
2. Go to **"Dataset Builder"** tab
3. Enter dataset in Audio Directory Path and click **Scan**
4. **Uncheck "All Instrumental"** (your songs have lyrics)
5. click **"Preprocess"**

### Training
6. Go to **"Train LoRA"** tab
7. Load your preprocessed dataset
8. Set: **Batch Size = 1**, **Save Every N Epochs = 100**, **Seed = 42**
9. Click **"Start Training"** and monitor the loss curve

### Generation
10. After training, go to the **Generate** tab

In [ ]:
!cd ACE-Step-1.5 && python3 -m acestep.acestep_v15_pipeline --share

Loaded configuration from /lab5/ACE-Step-1.5/.env.example (fallback)
2026-05-04 18:56:34.201 | DEBUG    | acestep.core.generation.handler.init_service_memory_basic:_apply_malloc_mmap_threshold:50 - [memory] Set M_MMAP_THRESHOLD=131072 for immediate OS reclaim of large frees
2026-05-04 18:56:40.511 | WARNING  | acestep.training.trainer:<module>:40 - bitsandbytes not installed. Using standard AdamW.
2026-05-04 18:56:43.328 | INFO     | acestep.gpu_config:get_gpu_memory_gb:578 - CUDA GPU detected: NVIDIA GeForce RTX 3090 (23.6 GB)

GPU Configuration Detected:
  GPU Memory: 23.57 GB
  Configuration Tier: tier6b
  Max Duration (with LM): 480s (8 min)
  Max Duration (without LM): 480s (8 min)
  Max Batch Size (with LM): 8
  Max Batch Size (without LM): 8
  Default LM Init: True
  Available LM Models: ['acestep-5Hz-lm-0.6B', 'acestep-5Hz-lm-1.7B', 'acestep-5Hz-lm-4B']

CPU offload disabled by default (GPU 23.6GB >= 20.0GB threshold)
Output directory: /lab5/ACE-Step-1.5/gradio_outputs
Creating

In [ ]:
prompts = [
    {
        "name": "new_folk_song_1",
        "prompt": "A lively Ukrainian folk dance song with accordion and tambourine, "
                  "humorous male vocals, fast polka rhythm, and a catchy "
                  "call-and-response chorus about village celebrations",
        "lyrics": (
            "[Verse 1]\n"
            "Ой на горі два дубки, а під ними козаки\n"
            "Козаченьки молоді, дивляться на дівочки\n"
            "\n"
            "[Chorus]\n"
            "Гуляй, гуляй, поки мати не лає\n"
            "Гуляй, гуляй, поки батько не знає\n"
            "Гуляй, гуляй, бо життя коротке\n"
            "А серденько молоде та палке"
        ),
        "duration": 90,
    },
    {
        "name": "new_folk_song_2",
        "prompt": "A moderate-tempo Ukrainian folk ballad with warm female vocals, "
                  "acoustic guitar and accordion, nostalgic melody about love, "
                  "with a gentle swaying waltz feel",
        "lyrics": (
            "[Verse 1]\n"
            "Місяць сяє над селом, тихо коло хати\n"
            "Вийшла дівчина з вінком, квіти поливати\n"
            "\n"
            "[Chorus]\n"
            "Ой, зоре моя вечірняя\n"
            "Покажи дорогу до кохання\n"
            "Ой, зоре моя ясная\n"
            "Підкажи де доля моя щасная"
        ),
        "duration": 75,
    },
    {
        "name": "new_folk_song_3",
        "prompt": "An upbeat humorous Ukrainian wedding folk song with energetic "
                  "male group vocals, fast accordion-driven melody, comedic lyrics "
                  "about a mother-in-law, and festive atmosphere",
        "lyrics": (
            "[Verse 1]\n"
            "Прийшов зять до тещі в гості, наварила тещенька борщу\n"
            "А він каже дай ще ковбаси, бо я більше їсти не хочу\n"
            "\n"
            "[Chorus]\n"
            "Ой тещенько, голубенько\n"
            "Нагодуй мене гарненько\n"
            "Ой тещенько, зіронько\n"
            "Налий чарку повненько"
        ),
        "duration": 60,
    },
]